# SMARD API — Live DA Price Fetch Check

**Question this notebook answers:** Can we, on demand, pull (a) the last N days of day-ahead prices and (b) tomorrow's prices once the EPEX auction has cleared, from the SMARD `chart_data` API?

**What we check:**
1. Past window — fetch the last `LOOKBACK_DAYS` of DA prices at the configured `RESOLUTION` and confirm every slot has a value.
2. Tomorrow window — try to fetch tomorrow's full price curve (Berlin local 00:00–24:00) and report how many slots are already published.

**Scope:** exploratory notebook only. Nothing under `src/` is touched.

## 1. Config

- **Filter 4169** = Deutschland/Luxemburg day-ahead price (EUR/MWh).
- **Region `DE`** covers the post-2018-10-01 DE/LU bidding zone.
- `LOOKBACK_DAYS` controls how far back from `now` to fetch.

In [1]:
import json
import urllib.request
from datetime import timedelta

import pandas as pd

BASE = "https://www.smard.de/app/chart_data"
FILTER = 4169          # DE/LU day-ahead price
REGION = "DE"
RESOLUTION = "hour"    # 'hour' or 'quarterhour'
LOOKBACK_DAYS = 7      # how many days of past prices to require

STEP = pd.Timedelta(hours=1) if RESOLUTION == "hour" else pd.Timedelta(minutes=15)
SLOT_LABEL = {"hour": "hour", "quarterhour": "quarter-hour"}[RESOLUTION]

now_utc = pd.Timestamp.now(tz="UTC")
now_berlin = now_utc.tz_convert("Europe/Berlin")
print(f"now (UTC):    {now_utc}")
print(f"now (Berlin): {now_berlin}")
print(f"resolution:   {RESOLUTION} ({STEP})")

now (UTC):    2026-04-23 12:09:34.184822+00:00
now (Berlin): 2026-04-23 14:09:34.184822+02:00
resolution:   hour (0 days 01:00:00)


## 2. Helpers — index + weekly bucket fetch

SMARD serves prices in weekly buckets. To cover any window we (1) read the index of available bucket timestamps, then (2) pull whichever buckets overlap the window we want.

In [2]:
def get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=15) as r:
        return json.loads(r.read())

def fetch_week(bucket_ms: int) -> pd.DataFrame:
    url = f"{BASE}/{FILTER}/{REGION}/{FILTER}_{REGION}_{RESOLUTION}_{bucket_ms}.json"
    p = get_json(url)
    df = pd.DataFrame(p["series"], columns=["ts_ms", "price_eur_mwh"])
    df["timestamp_utc"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    return df[["timestamp_utc", "price_eur_mwh"]]

def fetch_window(start_utc: pd.Timestamp, end_utc: pd.Timestamp) -> pd.DataFrame:
    """Fetch every weekly bucket overlapping [start_utc, end_utc] and slice."""
    index = get_json(f"{BASE}/{FILTER}/{REGION}/index_{RESOLUTION}.json")
    buckets = pd.to_datetime(index["timestamps"], unit="ms", utc=True)
    one_week = pd.Timedelta(days=7)
    needed = [
        int(b.timestamp() * 1000)
        for b in buckets
        if (b + one_week) > start_utc and b <= end_utc
    ]
    frames = [fetch_week(b) for b in needed]
    if not frames:
        return pd.DataFrame(columns=["timestamp_utc", "price_eur_mwh"])
    df = pd.concat(frames, ignore_index=True).sort_values("timestamp_utc")
    return df[(df.timestamp_utc >= start_utc) & (df.timestamp_utc < end_utc)].reset_index(drop=True)

## 3. Past window — last `LOOKBACK_DAYS` of prices

Confirms the API returns a complete, gap-free series for the lookback window. Every slot up to `now` should have a published price.

In [3]:
past_start = (now_utc - timedelta(days=LOOKBACK_DAYS)).floor(STEP)
past_end = now_utc.floor(STEP)
past = fetch_window(past_start, past_end)

expected_slots = int((past_end - past_start) / STEP)
got_rows = len(past)
n_nulls = past.price_eur_mwh.isna().sum()
n_negative = (past.price_eur_mwh < 0).sum()

W = 22
print(f"{'window:':<{W}}{past_start}  →  {past_end}")
print(f"{f'expected {SLOT_LABEL}s:':<{W}}{expected_slots}")
print(f"{'rows returned:':<{W}}{got_rows}")
print(f"{'nulls in window:':<{W}}{n_nulls}")
print(f"{'negative prices:':<{W}}{n_negative}")
print()
print("head:"); print(past.head())
print("tail:"); print(past.tail())

past_ok = (got_rows == expected_slots) and (n_nulls == 0)
print(f"\npast window OK: {past_ok}")

window:               2026-04-16 12:00:00+00:00  →  2026-04-23 12:00:00+00:00
expected hours:       168
rows returned:        168
nulls in window:      0
negative prices:      20

head:
              timestamp_utc  price_eur_mwh
0 2026-04-16 12:00:00+00:00          53.97
1 2026-04-16 13:00:00+00:00          70.87
2 2026-04-16 14:00:00+00:00          89.76
3 2026-04-16 15:00:00+00:00         111.56
4 2026-04-16 16:00:00+00:00         131.15
tail:
                timestamp_utc  price_eur_mwh
163 2026-04-23 07:00:00+00:00          63.29
164 2026-04-23 08:00:00+00:00          -0.56
165 2026-04-23 09:00:00+00:00          -6.18
166 2026-04-23 10:00:00+00:00         -29.01
167 2026-04-23 11:00:00+00:00         -49.56

past window OK: True


## 4. Tomorrow window — next day-ahead prices

EPEX clears the day-ahead auction around 12:45 CET. SMARD mirrors the result with some delay. This cell tries to fetch tomorrow's full price curve (00:00–24:00 Berlin local) and reports how many slots are already published. With `RESOLUTION="hour"` that's 24 slots; with `"quarterhour"` it's 96.

In [4]:
tomorrow_start_local = (now_berlin.normalize() + pd.Timedelta(days=1))
tomorrow_end_local = tomorrow_start_local + pd.Timedelta(days=1)
tomorrow_start_utc = tomorrow_start_local.tz_convert("UTC")
tomorrow_end_utc = tomorrow_end_local.tz_convert("UTC")

tomorrow = fetch_window(tomorrow_start_utc, tomorrow_end_utc)

expected = int((tomorrow_end_utc - tomorrow_start_utc) / STEP)
published = tomorrow.price_eur_mwh.notna().sum()
missing = tomorrow.price_eur_mwh.isna().sum() + (expected - len(tomorrow))

W = 26
print(f"{'tomorrow (Berlin local):':<{W}}{tomorrow_start_local.date()}")
print(f"{'window (UTC):':<{W}}{tomorrow_start_utc} → {tomorrow_end_utc}")
print(f"{f'expected {SLOT_LABEL}s:':<{W}}{expected}")
print(f"{'published:':<{W}}{published}")
print(f"{'still missing:':<{W}}{missing}")

if published == expected:
    print("\nauction has cleared and full next-day curve is available:")
    print(tomorrow.set_index("timestamp_utc")["price_eur_mwh"])
elif published == 0:
    print("\nauction not yet published — re-run after ~14:00 CET")
else:
    print("\npartial publication (unexpected — usually the auction goes live all-or-nothing):")
    print(tomorrow)

tomorrow_ok = (published == expected)
print(f"\ntomorrow window OK: {tomorrow_ok}")

tomorrow (Berlin local):  2026-04-24
window (UTC):             2026-04-23 22:00:00+00:00 → 2026-04-24 22:00:00+00:00
expected hours:           24
published:                24
still missing:            0

auction has cleared and full next-day curve is available:
timestamp_utc
2026-04-23 22:00:00+00:00    102.46
2026-04-23 23:00:00+00:00    100.13
2026-04-24 00:00:00+00:00    100.71
2026-04-24 01:00:00+00:00    101.69
2026-04-24 02:00:00+00:00    105.36
2026-04-24 03:00:00+00:00    116.31
2026-04-24 04:00:00+00:00    131.87
2026-04-24 05:00:00+00:00    137.99
2026-04-24 06:00:00+00:00    117.68
2026-04-24 07:00:00+00:00     80.91
2026-04-24 08:00:00+00:00     19.57
2026-04-24 09:00:00+00:00     -0.04
2026-04-24 10:00:00+00:00     -6.91
2026-04-24 11:00:00+00:00    -26.14
2026-04-24 12:00:00+00:00    -30.00
2026-04-24 13:00:00+00:00    -16.56
2026-04-24 14:00:00+00:00     -1.61
2026-04-24 15:00:00+00:00     50.91
2026-04-24 16:00:00+00:00    104.82
2026-04-24 17:00:00+00:00    133.81
2026

## 5. Verdict

- **Past window OK** → SMARD can be polled at any time to backfill the last `LOOKBACK_DAYS` of realised DA prices.
- **Tomorrow window OK** → tomorrow's auction is already published and we can already feed those 24 prices into a forward-looking MILP run.
- If `tomorrow_ok` is False before ~13:00 CET, that is the expected EPEX schedule, not a SMARD failure. Re-run after the auction clears.

In [5]:
print(f"past_ok     = {past_ok}")
print(f"tomorrow_ok = {tomorrow_ok}")

past_ok     = True
tomorrow_ok = True
